# KDE + DBSCAN — Hubs terciarios

Agrega hubs **terciarios** (zonas de carga sin cobertura KDE) al resultado de `kde_clusters.ipynb`.

**Entradas:**
- `results/readykde.csv` — hubs primarios y secundarios del paso anterior
- `coordenadas3/*.parquet` — coordenadas de carga limpias por estado

**Salida:** `results/final_results.csv` — lista unificada primaria / secundaria / terciaria

**Parámetro clave:** `RADIUS_MILES` — radio de exclusion para hubs terciarios respecto a hubs KDE ya existentes.

## Imports

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import folium
import networkx as nx
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors, BallTree
from math import radians, sin, cos, sqrt, atan2

from metricas import coordenadas, cargasinout, haversine, aplicardbscan, df_prueba

## Parámetros

In [2]:
DBSCAN_EPS_MI  = 20    # radio DBSCAN en millas
DBSCAN_MIN_PTS = 0.75  # percentil mínimo de cargas para puntos DBSCAN (pasado a df_prueba)

cols = [
    'ID', 'Posted', 'PickUp', 'CityOrigin', 'CityDestination',
    'Distance', 'Equip', 'StateOrigin', 'HubOrigin',
    'StateDestination', 'HubDestination', 'Broker_Shipper', 'RatePerMile',
    'LatOrigin', 'LngOrigin', 'LatDestination', 'LngDestination'
]

## Datos de entrada

In [3]:
kde_hubs = pd.read_csv('../4. kde/results/clusters_kde.csv')
hubs_ref  = pd.read_excel('../utils/hubs_id.xlsx')

print('Hubs KDE:', kde_hubs.shape)
print(kde_hubs['categoria'].value_counts() if 'categoria' in kde_hubs.columns else kde_hubs.get('categoría', pd.Series()).value_counts())

Hubs KDE: (88, 8)
categoría
primaria      57
secundaria    31
Name: count, dtype: int64


## Funciones

In [4]:
EARTH_RADIUS_MILES = 3958.8
MILES_TO_RADIANS   = 1 / EARTH_RADIUS_MILES

def densest_point_by_loads(df, cluster_col='cluster', lat_col='lat', lon_col='lng',
                           load_col='cargas', radius_miles=5):
    """Selecciona, por cluster DBSCAN, el punto cuya vecindad local tiene mayor suma de cargas."""
    results = []
    for c in df[cluster_col].unique():
        if c == -1:
            continue
        subset = df[df[cluster_col] == c].copy()
        coords     = subset[[lat_col, lon_col]].to_numpy()
        loads      = subset[load_col].to_numpy().astype(float)
        coords_rad = np.radians(coords)
        radius_rad = radius_miles * MILES_TO_RADIANS

        nbrs = NearestNeighbors(radius=radius_rad, algorithm='ball_tree', metric='haversine')
        nbrs.fit(coords_rad)
        _, indices = nbrs.radius_neighbors(coords_rad)

        best_score, best_idx = -np.inf, None
        for i, neigh in enumerate(indices):
            score = loads[neigh].sum()
            if score > best_score:
                best_score, best_idx = score, i

        center_lat, center_lon = coords[best_idx]
        results.append({
            'cluster': c,
            'lat': center_lat,
            'lng': center_lon,
            'total_loads_local': best_score,
            'cluster_size': len(subset)
        })
    return pd.DataFrame(results)


def crear_hubs_terciarios(kde_hubs, centroids_dbscan, state, radius_miles):
    """
    Filtra centroides DBSCAN que no estén dentro de `radius_miles` de ningún hub KDE del estado
    ni de ningún terciario ya aceptado.
    """
    radius_rad = radius_miles / EARTH_RADIUS_MILES
    cc = kde_hubs[kde_hubs['state'] == state]
    if cc.empty:
        centroids_dbscan = centroids_dbscan.copy()
        centroids_dbscan['categoría'] = 'terciario'
        centroids_dbscan['state'] = state
        return centroids_dbscan

    coords_old = np.radians(cc[['lat', 'lng']].to_numpy())
    tree_old   = BallTree(coords_old, metric='haversine')

    centroids_new  = centroids_dbscan.sort_values('total_loads_local', ascending=True).reset_index(drop=True)
    kept_new       = []
    accepted_coords = []

    for _, row in centroids_new.iterrows():
        point = np.radians([[row['lat'], row['lng']]])
        if len(tree_old.query_radius(point, r=radius_rad)[0]) > 0:
            continue
        if accepted_coords:
            tree_new = BallTree(np.array(accepted_coords), metric='haversine')
            if len(tree_new.query_radius(point, r=radius_rad)[0]) > 0:
                continue
        kept_new.append(row)
        accepted_coords.append(point[0])

    df_kept = pd.DataFrame(kept_new)
    if df_kept.empty:
        return cc.copy()

    df_kept['categoría'] = 'terciario'
    df_kept['state'] = state
    return pd.concat([cc, df_kept], ignore_index=True)


def limpiar_hubs_terciarios_global(df, radius_miles):
    """
    Pase global: elimina terciarios dentro de `radius_miles` de cualquier hub no-terciario,
    y dedup entre terciarios cercanos (conserva el de mayor total_loads_local).
    """
    def hav(lat1, lon1, lat2, lon2):
        R = 3958.8
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat, dlon = lat2 - lat1, lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        return 2 * R * np.arcsin(np.sqrt(a))

    terciarios = df[df['categoría'] == 'terciario'].reset_index(drop=True)
    otros      = df[df['categoría'] != 'terciario'].reset_index(drop=True)
    to_remove  = set()

    for i, rt in terciarios.iterrows():
        for _, ro in otros.iterrows():
            if hav(rt['lat'], rt['lng'], ro['lat'], ro['lng']) <= radius_miles:
                to_remove.add(i)
                break

    for i, rt in terciarios.iterrows():
        if i in to_remove:
            continue
        vecinos = [j for j, ro in terciarios.iterrows()
                   if i != j and j not in to_remove
                   and hav(rt['lat'], rt['lng'], ro['lat'], ro['lng']) <= radius_miles]
        if vecinos:
            grupo = [i] + vecinos
            max_idx = terciarios.loc[grupo, 'total_loads_local'].idxmax()
            to_remove.update(idx for idx in grupo if idx != max_idx)

    terciarios_ok = terciarios.drop(index=to_remove)
    print(f'Terciarios originales: {len(terciarios)} | eliminados: {len(to_remove)} | restantes: {len(terciarios_ok)}')
    return pd.concat([otros, terciarios_ok], ignore_index=True)

## Etapa 1 — DBSCAN por estado y generación de terciarios candidatos

Por cada estado, aplica DBSCAN sobre `coordenadas3/`, selecciona el centroide más denso por cluster y descarta los que caen dentro del radio de exclusión de hubs KDE existentes.

In [5]:
carpeta  = '../3. cleaning_coords/coordenadas'
archivos = sorted(glob.glob(os.path.join(carpeta, '*.parquet')))
centroids_total = pd.DataFrame()

for i, archivo in enumerate(archivos, 1):
    df    = pd.read_parquet(archivo)
    state = os.path.splitext(os.path.basename(archivo))[0]
    print(i, state)

    loads  = cargasinout(df, state, cols)
    coords = coordenadas(df, state)

    coords, _ = aplicardbscan(pd.DataFrame(), coords, state, 10)
    clusters  = df_prueba(df, loads, coords, pd.DataFrame(), state, 0.75)

    centroids = densest_point_by_loads(
        clusters, cluster_col='cluster', lat_col='lat', lon_col='lng',
        load_col='cargas', radius_miles=5
    )

    pre_hubs = crear_hubs_terciarios(kde_hubs, centroids, state, 50)
    centroids_total = pd.concat([centroids_total, pre_hubs], ignore_index=True)

os.makedirs('results', exist_ok=True)
#centroids_total.to_csv('results/all_centroides.csv', index=False)
#print('Guardado: results/all_centroides.csv |', centroids_total.shape)

1 AL
Eps sugerido (percentil 90): 0.0711
(13751, 6)
(12168, 6)


/home/karen/hubs/jupyter-env/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


2 AR
Eps sugerido (percentil 90): 0.0803
(9798, 6)
(8590, 6)


/home/karen/hubs/jupyter-env/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


3 AZ
Eps sugerido (percentil 90): 0.0431
(12429, 6)
(10801, 6)


/home/karen/hubs/jupyter-env/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


... (86 outputs left out)

47 WV
Eps sugerido (percentil 90): 0.1032
(5059, 6)
(4483, 6)


/home/karen/hubs/jupyter-env/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


48 WY
Eps sugerido (percentil 90): 0.1205
(2237, 6)
(1898, 6)


/home/karen/hubs/jupyter-env/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


## Etapa 2 — Limpieza global de terciarios

Elimina terciarios solapados con hubs KDE o entre sí (a nivel nacional).

In [6]:
df_final_centroides = limpiar_hubs_terciarios_global(centroids_total, 75)
print('Total después de limpieza global:', df_final_centroides.shape)

Terciarios originales: 290 | eliminados: 186 | restantes: 104
Total después de limpieza global: (192, 11)


In [7]:
print(df_final_centroides['categoria'].value_counts() if 'categoria' in df_final_centroides.columns else df_final_centroides.get('categoría', pd.Series()).value_counts())

categoría
terciario     104
primaria       57
secundaria     31
Name: count, dtype: int64


In [8]:
df_final_centroides

,id,lat,lng,Z,new_name,state,fam_id,categoría,cluster,total_loads_local,cluster_size
0,1.0,33.455575,-86.839677,1.000000,Pico AL,AL,familia_14,primaria,NaN,NaN,NaN
1,69.0,30.718925,-88.106887,0.583708,Pico AL,AL,familia_12,secundaria,NaN,NaN,NaN
2,80.0,32.399324,-86.311673,0.436066,Pico AL,AL,familia_18,secundaria,NaN,NaN,NaN
3,3.0,34.739338,-92.289498,1.000000,Pico AR,AR,familia_11,primaria,NaN,NaN,NaN
4,4.0,36.218294,-94.151811,0.894564,Pico AR,AR,familia_13,primaria,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
187,NaN,44.903450,-90.306580,NaN,NaN,WI,NaN,terciario,13.0,1172.0,120.0
188,NaN,43.864399,-91.253303,NaN,NaN,WI,NaN,terciario,9.0,4051.0,354.0
189,NaN,42.268190,-104.734121,NaN,NaN,WY,NaN,terciario,8.0,649.0,62.0
190,NaN,43.500000,-110.850000,NaN,NaN,WY,NaN,terciario,5.0,1088.0,140.0


In [9]:
df_final_centroides.loc[df_final_centroides['categoría']=='terciaria'].to_csv('clusters_terciarios.csv')

## Etapa 4 — Exportar `final_results.csv`

In [10]:
df_final_centroides.to_csv('results/final_results.csv', index=False)
print('Guardado: results/final_results.csv')
df_final_centroides.groupby('categoría')['state'].count()

Guardado: results/final_results.csv


categoría
primaria       57
secundaria     31
terciario     104
Name: state, dtype: int64

## Visualización final

In [11]:
cat_colors = {'primaria': 'red', 'secundaria': 'blue', 'terciario': 'green'}
center = [df_final_centroides['lat'].mean(), df_final_centroides['lng'].mean()]
m = folium.Map(location=[39.5, -98.35], zoom_start=5, tiles="OpenStreetMap")

for _, row in df_final_centroides.iterrows():
    cat   = row.get('categoría', 'terciario')
    color = cat_colors.get(cat, 'gray')
    popup = f"{row.get('new_name', '')} ({row.get('state', '')}) | {cat}"
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=5 if cat == 'terciario' else 7,
        color=color, fill=True, fill_color=color, fill_opacity=0.8,
        popup=popup
    ).add_to(m)

legend = (
    '<div style="position:fixed;bottom:30px;left:30px;width:180px;'
    'background:white;border:2px solid grey;border-radius:8px;padding:10px;z-index:9999;font-size:14px">'
    '<b>Tipo de hub</b><br>'
    '<span style="color:red;font-size:18px">&#9679;</span> Primaria<br>'
    '<span style="color:blue;font-size:18px">&#9679;</span> Secundaria<br>'
    '<span style="color:green;font-size:18px">&#9679;</span> Terciaria</div>'
)
m.get_root().html.add_child(folium.Element(legend))
m

## Comparación con hubs de referencia

In [12]:
h = (df_final_centroides.groupby('state').size()
       .reset_index(name='nuevos')
       .merge(hubs_ref.groupby('state')['new_name'].count().reset_index(name='referencia'),
              on='state', how='outer')
       .sort_values('nuevos', ascending=False))
h['diff'] = h['nuevos'] - h['referencia']
h

,state,nuevos,referencia,diff
40,TX,13,13.0,0.0
3,CA,9,7.0,2.0
23,MT,8,4.0,4.0
7,FL,7,7.0,0.0
25,ND,6,3.0,3.0
22,MS,6,5.0,1.0
8,GA,6,6.0,0.0
47,WY,6,3.0,3.0
9,IA,5,6.0,-1.0
44,WA,5,4.0,1.0
